# Live Glucose Alarm Detection with Temporal Aggregation

This notebook performs **real-time audio classification** with temporal voting for robust alarm detection.

## Features

- 🎯 **Temporal Aggregation**: Votes across multiple consecutive windows (reduces false alarms)
- 🔧 **Auto-Configuration**: Automatically loads audio parameters from model metadata
- 🔄 **Multi-Window Support**: Works with models trained on any window size (1.0s, 1.5s, 2.0s, etc.)
- ⚙️ **Adjustable Thresholds**: Tune sensitivity vs. false alarm rate
- 📊 **Real-time Visualization**: See predictions and confidence in real-time
- 🎤 **Live Audio Capture**: Records from microphone using model's window size

## How It Works

```
Audio Stream → 1s Windows → Mel-Spectrogram → CNN → Probability → Temporal Voting → Alert
```

### Temporal Voting Strategy

Instead of alerting on a single window prediction:
```python
# Collect last N predictions
predictions = [0.2, 0.3, 0.8, 0.9, 0.7]  # Last 5 windows

# Alert if majority vote "alarm"
if sum(p > threshold for p in predictions) >= vote_threshold:
    TRIGGER_ALERT()
```

**Benefits**:
- ✅ Reduces false alarms (need multiple windows to agree)
- ✅ More robust to noise
- ✅ Still fast (< 2 second latency)

## Configuration

You can adjust:
- `MODEL_PATH`: Which model to load from `models/`
- `CONFIDENCE_THRESHOLD`: Probability threshold (0.0-1.0)
- `WINDOW_SIZE`: How many recent predictions to consider
- `VOTE_THRESHOLD`: How many windows must agree to trigger alert

## 1. Imports & Configuration

In [1]:
# Imports
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import sounddevice as sd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import deque
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports successful")

✓ Imports successful


In [11]:
# ==================== CONFIGURATION ====================

# Model Configuration
MODEL_PATH = "models/glucose_alarm_cnn_w1.5s_20260315_184214.pth"  # ← Latest model with ref=1.0 fix!
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Audio Configuration
# NOTE: These will be AUTO-LOADED from model metadata if available!
# Only used as fallback if metadata file is missing.
SAMPLE_RATE = 16000
WINDOW_DURATION = 1.5  # seconds (fallback - will be loaded from metadata)
N_MELS = 64
N_FFT = 1024           # fallback - will be loaded from metadata
HOP_LENGTH = 256       # fallback - will be loaded from metadata

# Temporal Aggregation Configuration
CONFIDENCE_THRESHOLD = 0.9    # Probability threshold (0.0-1.0)
TEMPORAL_WINDOW_SIZE = 5      # Number of recent predictions to consider
VOTE_THRESHOLD = 3            # How many windows must agree (out of TEMPORAL_WINDOW_SIZE)

# Alert Configuration
ALERT_COOLDOWN = 5.0          # Seconds between alerts (prevent spam)

# ======================================================

print("Configuration (before loading metadata):")
print(f"  Model: {MODEL_PATH}")
print(f"  Device: {DEVICE}")
print(f"  Window Duration: {WINDOW_DURATION}s (may be updated from metadata)")
print(f"  Confidence Threshold: {CONFIDENCE_THRESHOLD}")
print(f"  Temporal Window: {TEMPORAL_WINDOW_SIZE} predictions")
print(f"  Vote Threshold: {VOTE_THRESHOLD}/{TEMPORAL_WINDOW_SIZE} windows must agree")
print(f"  Alert Cooldown: {ALERT_COOLDOWN}s")

Configuration (before loading metadata):
  Model: models/glucose_alarm_cnn_w1.5s_20260315_184214.pth
  Device: cpu
  Window Duration: 1.5s (may be updated from metadata)
  Confidence Threshold: 0.9
  Temporal Window: 5 predictions
  Vote Threshold: 3/5 windows must agree
  Alert Cooldown: 5.0s


## 2. Load Model Metadata (Auto-Configure Audio Parameters)

This automatically loads the correct audio parameters from the model's metadata file.

In [12]:
import json

# Try to load metadata file
metadata_path = Path(MODEL_PATH.replace('.pth', '_metadata.json'))

if metadata_path.exists():
    print(f"✓ Found metadata file: {metadata_path.name}")
    
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    # Update audio parameters from metadata
    audio_params = metadata['audio_params']
    SAMPLE_RATE = audio_params['sample_rate']
    WINDOW_DURATION = audio_params['window_length']
    N_MELS = audio_params['n_mels']
    N_FFT = audio_params['n_fft']
    HOP_LENGTH = audio_params['hop_length']
    
    print("\n✓ Audio parameters loaded from metadata:")
    print(f"  Sample Rate: {SAMPLE_RATE} Hz")
    print(f"  Window Duration: {WINDOW_DURATION}s")
    print(f"  Mel Bands: {N_MELS}")
    print(f"  FFT Size: {N_FFT}")
    print(f"  Hop Length: {HOP_LENGTH}")
    
    # Display model performance
    if 'metrics' in metadata:
        metrics = metadata['metrics']
        print("\n📊 Model Performance:")
        print(f"  Accuracy:  {metrics['accuracy']*100:.2f}%")
        print(f"  Recall:    {metrics['recall']*100:.2f}%")
        print(f"  Precision: {metrics['precision']*100:.2f}%")
        print(f"  F1 Score:  {metrics['f1_score']*100:.2f}%")
else:
    print(f"⚠️  Metadata file not found: {metadata_path.name}")
    print("⚠️  Using fallback audio parameters from configuration cell")
    print("⚠️  This may cause incorrect predictions if parameters don't match training!")
    print("\nUsing:")
    print(f"  Sample Rate: {SAMPLE_RATE} Hz")
    print(f"  Window Duration: {WINDOW_DURATION}s")
    print(f"  Mel Bands: {N_MELS}")
    print(f"  FFT Size: {N_FFT}")
    print(f"  Hop Length: {HOP_LENGTH}")

✓ Found metadata file: glucose_alarm_cnn_w1.5s_20260315_184214_metadata.json

✓ Audio parameters loaded from metadata:
  Sample Rate: 16000 Hz
  Window Duration: 1.5s
  Mel Bands: 64
  FFT Size: 1024
  Hop Length: 256

📊 Model Performance:
  Accuracy:  90.03%
  Recall:    93.01%
  Precision: 87.79%
  F1 Score:  90.32%


## 3. Model Definition

This must match the architecture used during training.

In [13]:
class GlucoseAlarmCNN(nn.Module):
    """
    Convolutional Neural Network for glucose alarm detection.
    
    Input: (batch_size, 1, n_mels, time_frames)
    Output: (batch_size, 1) - single logit for binary classification
    """
    
    def __init__(self, n_mels=64):
        super(GlucoseAlarmCNN, self).__init__()
        
        # Convolutional block 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Convolutional block 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Convolutional block 3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Global average pooling
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully connected layer
        self.fc = nn.Linear(128, 1)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        """
        Forward pass.
        
        Parameters:
        -----------
        x : torch.Tensor
            Input tensor of shape (batch_size, 1, n_mels, time_frames)
        
        Returns:
        --------
        torch.Tensor
            Output logits of shape (batch_size, 1)
        """
        # Block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        
        # Block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        
        # Block 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        
        # Global average pooling
        x = self.global_avg_pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Dropout
        x = self.dropout(x)
        
        # Fully connected
        x = self.fc(x)
        
        return x

print("✓ Model class defined")

✓ Model class defined


## 4. Load Model

In [14]:
# Check if model file exists
model_path = Path(MODEL_PATH)
if not model_path.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

# Initialize model
model = GlucoseAlarmCNN(n_mels=N_MELS)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()  # Set to evaluation mode

print(f"✓ Model loaded from: {MODEL_PATH}")
print(f"✓ Model on device: {DEVICE}")
print(f"✓ Model in eval mode")

✓ Model loaded from: models/glucose_alarm_cnn_w1.5s_20260315_184214.pth
✓ Model on device: cpu
✓ Model in eval mode


## 5. Audio Processing Functions

In [15]:
def audio_to_melspectrogram(audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """
    Convert audio to mel-spectrogram.
    
    Parameters:
    -----------
    audio : np.ndarray
        Audio signal
    sr : int
        Sample rate
    n_mels : int
        Number of mel bands
    n_fft : int
        FFT size
    hop_length : int
        Hop length
    
    Returns:
    --------
    torch.Tensor
        Mel-spectrogram tensor of shape (1, 1, n_mels, time_frames)
    """
    # Compute mel-spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        n_fft=n_fft,
        hop_length=hop_length
    )
    
    # Convert to log scale (dB)
    # Use ref=np.max for pure pattern recognition (volume-independent)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Compute RMS energy to detect TRUE silence (not just quiet audio)
    rms_energy = np.sqrt(np.mean(audio**2))
    
    # Only penalize EXTREME silence (background noise when nothing is happening)
    # Lowered threshold to 0.0001 to avoid penalizing quiet alarms
    if rms_energy < 0.0001:  # Extreme silence only
        mel_spec_db = mel_spec_db - 60  # Heavy penalty for true silence
    
    # Convert to tensor and add batch + channel dimensions
    mel_tensor = torch.FloatTensor(mel_spec_db).unsqueeze(0).unsqueeze(0)
    
    return mel_tensor


def predict_window(audio, model, device=DEVICE):
    """
    Predict alarm probability for a single audio window.
    
    Parameters:
    -----------
    audio : np.ndarray
        Audio signal (1 second)
    model : nn.Module
        Trained model
    device : torch.device
        Device to run inference on
    
    Returns:
    --------
    float
        Probability of alarm (0.0 - 1.0)
    """
    # Convert to mel-spectrogram
    mel_spec = audio_to_melspectrogram(audio)
    mel_spec = mel_spec.to(device)
    
    # Predict
    with torch.no_grad():
        logit = model(mel_spec)
        probability = torch.sigmoid(logit).item()
    
    return probability

print("✓ Audio processing functions defined")

✓ Audio processing functions defined


## 6. Temporal Aggregation Class

In [16]:
class TemporalAggregator:
    """
    Aggregates predictions over time using a sliding window voting mechanism.
    
    This reduces false alarms by requiring multiple consecutive windows to agree.
    """
    
    def __init__(self, window_size=5, vote_threshold=3, confidence_threshold=0.5, cooldown=5.0):
        """
        Parameters:
        -----------
        window_size : int
            Number of recent predictions to consider
        vote_threshold : int
            How many predictions must exceed confidence_threshold to trigger alert
        confidence_threshold : float
            Probability threshold for individual predictions (0.0-1.0)
        cooldown : float
            Minimum seconds between alerts
        """
        self.window_size = window_size
        self.vote_threshold = vote_threshold
        self.confidence_threshold = confidence_threshold
        self.cooldown = cooldown
        
        self.predictions = deque(maxlen=window_size)
        self.last_alert_time = 0
        self.alert_count = 0
    
    def add_prediction(self, probability):
        """
        Add a new prediction and check if alert should be triggered.
        
        Parameters:
        -----------
        probability : float
            Alarm probability from model (0.0-1.0)
        
        Returns:
        --------
        dict
            {
                'alert': bool,           # Should trigger alert?
                'probability': float,    # Current probability
                'vote_count': int,       # How many windows voted "alarm"
                'confidence': float      # Average confidence of recent predictions
            }
        """
        self.predictions.append(probability)
        
        # Count how many recent predictions exceed threshold
        vote_count = sum(1 for p in self.predictions if p > self.confidence_threshold)
        
        # Calculate average confidence
        avg_confidence = np.mean(list(self.predictions)) if self.predictions else 0.0
        
        # Check if should trigger alert
        should_alert = False
        current_time = time.time()
        
        if vote_count >= self.vote_threshold:
            # Check cooldown
            if current_time - self.last_alert_time >= self.cooldown:
                should_alert = True
                self.last_alert_time = current_time
                self.alert_count += 1
        
        return {
            'alert': should_alert,
            'probability': probability,
            'vote_count': vote_count,
            'confidence': avg_confidence
        }
    
    def reset(self):
        """Reset the aggregator state."""
        self.predictions.clear()
        self.last_alert_time = 0

print("✓ Temporal aggregator class defined")

✓ Temporal aggregator class defined


## 7. Test on Sample Audio File

Before running live inference, let's test on a sample file to verify everything works.

In [17]:
# Test on a sample file
test_file = "dataset/train/20260111_204751_window_0000.wav"  # ← Change this to test different files

if Path(test_file).exists():
    print(f"Testing on: {test_file}\n")
    
    # Load audio
    audio, sr = librosa.load(test_file, sr=SAMPLE_RATE, mono=True)
    
    # Predict
    probability = predict_window(audio, model)
    
    print(f"Prediction Results:")
    print(f"  Probability: {probability:.4f}")
    print(f"  Classification: {'🚨 ALARM' if probability > CONFIDENCE_THRESHOLD else '✓ Non-alarm'}")
    print(f"  Confidence: {max(probability, 1-probability):.1%}")
    
    # Visualize mel-spectrogram
    mel_spec = audio_to_melspectrogram(audio)
    
    plt.figure(figsize=(10, 4))
    plt.imshow(mel_spec[0, 0].numpy(), aspect='auto', origin='lower', cmap='viridis')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Mel-Spectrogram | Probability: {probability:.4f}')
    plt.xlabel('Time Frames')
    plt.ylabel('Mel Bands')
    plt.tight_layout()
    plt.show()
else:
    print(f"Test file not found: {test_file}")
    print("Skipping test...")

Test file not found: dataset/train/20260111_204751_window_0000.wav
Skipping test...


## 8. Live Audio Inference

**⚠️ WARNING**: This will start recording from your microphone!

### How to Use:

1. **Run the cell below** to start live monitoring
2. **Make some noise** (or play an alarm sound)
3. **Watch the output** for predictions and alerts
4. **Press Ctrl+C** in the output to stop

### What You'll See:

```
[12:34:56] Prob: 0.23 | Votes: 1/5 | Avg: 0.18 | ✓ No alarm
[12:34:57] Prob: 0.87 | Votes: 2/5 | Avg: 0.45 | ⚠️ Building...
[12:34:58] Prob: 0.92 | Votes: 3/5 | Avg: 0.67 | 🚨 ALARM DETECTED!
```

In [18]:
def live_inference(duration=60):
    """
    Run live audio inference.
    
    Parameters:
    -----------
    duration : int
        How long to run (seconds). Set to None for infinite.
    """
    # Initialize aggregator
    aggregator = TemporalAggregator(
        window_size=TEMPORAL_WINDOW_SIZE,
        vote_threshold=VOTE_THRESHOLD,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        cooldown=ALERT_COOLDOWN
    )
    
    print("="*70)
    print("🎤 LIVE AUDIO MONITORING STARTED")
    print("="*70)
    print(f"Configuration:")
    print(f"  Window Size: {TEMPORAL_WINDOW_SIZE} predictions")
    print(f"  Vote Threshold: {VOTE_THRESHOLD}/{TEMPORAL_WINDOW_SIZE} must agree")
    print(f"  Confidence Threshold: {CONFIDENCE_THRESHOLD}")
    print(f"  Alert Cooldown: {ALERT_COOLDOWN}s")
    print(f"  Duration: {duration}s" if duration else "  Duration: Infinite (Ctrl+C to stop)")
    print("="*70)
    print()
    
    start_time = time.time()
    window_count = 0
    
    try:
        while True:
            # Check duration
            if duration and (time.time() - start_time) >= duration:
                break
            
            # Record 1 second of audio
            audio = sd.rec(
                int(WINDOW_DURATION * SAMPLE_RATE),
                samplerate=SAMPLE_RATE,
                channels=1,
                dtype='float32'
            )
            sd.wait()  # Wait for recording to finish
            audio = audio.flatten()
            
            # Predict
            probability = predict_window(audio, model)
            
            # Add to aggregator
            result = aggregator.add_prediction(probability)
            
            # Format output
            timestamp = datetime.now().strftime("%H:%M:%S")
            vote_str = f"{result['vote_count']}/{TEMPORAL_WINDOW_SIZE}"
            
            # Status indicator
            if result['alert']:
                status = "🚨 ALARM DETECTED!"
                print(f"\n{'='*70}")
                print(f"[{timestamp}] {status}")
                print(f"  Probability: {probability:.4f}")
                print(f"  Votes: {vote_str}")
                print(f"  Avg Confidence: {result['confidence']:.4f}")
                print(f"  Total Alerts: {aggregator.alert_count}")
                print(f"{'='*70}\n")
            elif result['vote_count'] >= VOTE_THRESHOLD - 1:
                status = "⚠️ Building..."
                print(f"[{timestamp}] Prob: {probability:.2f} | Votes: {vote_str} | Avg: {result['confidence']:.2f} | {status}")
            else:
                status = "✓ No alarm"
                print(f"[{timestamp}] Prob: {probability:.2f} | Votes: {vote_str} | Avg: {result['confidence']:.2f} | {status}")
            
            window_count += 1
    
    except KeyboardInterrupt:
        print("\n\n⏹️ Stopped by user")
    
    # Summary
    elapsed = time.time() - start_time
    print("\n" + "="*70)
    print("📊 SESSION SUMMARY")
    print("="*70)
    print(f"  Duration: {elapsed:.1f}s")
    print(f"  Windows Processed: {window_count}")
    print(f"  Total Alerts: {aggregator.alert_count}")
    print(f"  Alert Rate: {aggregator.alert_count / (elapsed / 60):.2f} alerts/minute")
    print("="*70)

print("✓ Live inference function defined")
print("\n💡 Run: live_inference(duration=60) to start monitoring for 60 seconds")
print("💡 Run: live_inference(duration=None) for infinite monitoring (Ctrl+C to stop)")

✓ Live inference function defined

💡 Run: live_inference(duration=60) to start monitoring for 60 seconds
💡 Run: live_inference(duration=None) for infinite monitoring (Ctrl+C to stop)


In [19]:
live_inference()

🎤 LIVE AUDIO MONITORING STARTED
Configuration:
  Window Size: 5 predictions
  Vote Threshold: 3/5 must agree
  Confidence Threshold: 0.9
  Alert Cooldown: 5.0s
  Duration: 60s

[19:15:40] Prob: 0.03 | Votes: 0/5 | Avg: 0.03 | ✓ No alarm
[19:15:42] Prob: 0.37 | Votes: 0/5 | Avg: 0.20 | ✓ No alarm
[19:15:44] Prob: 0.09 | Votes: 0/5 | Avg: 0.16 | ✓ No alarm
[19:15:46] Prob: 0.28 | Votes: 0/5 | Avg: 0.19 | ✓ No alarm
[19:15:47] Prob: 0.03 | Votes: 0/5 | Avg: 0.16 | ✓ No alarm
[19:15:49] Prob: 0.04 | Votes: 0/5 | Avg: 0.16 | ✓ No alarm
[19:15:51] Prob: 0.07 | Votes: 0/5 | Avg: 0.10 | ✓ No alarm
[19:15:53] Prob: 0.00 | Votes: 0/5 | Avg: 0.08 | ✓ No alarm
[19:15:54] Prob: 0.00 | Votes: 0/5 | Avg: 0.03 | ✓ No alarm
[19:15:56] Prob: 0.09 | Votes: 0/5 | Avg: 0.04 | ✓ No alarm
[19:15:58] Prob: 0.49 | Votes: 0/5 | Avg: 0.13 | ✓ No alarm
[19:16:00] Prob: 0.00 | Votes: 0/5 | Avg: 0.12 | ✓ No alarm
[19:16:01] Prob: 0.00 | Votes: 0/5 | Avg: 0.12 | ✓ No alarm
[19:16:03] Prob: 0.00 | Votes: 0/5 | Avg: 0

## 9. Batch Testing on Multiple Files

Test the model on multiple files to see how temporal aggregation performs.

In [9]:
def test_on_files(file_paths, show_plot=True):
    """
    Test temporal aggregation on a sequence of audio files.
    
    Parameters:
    -----------
    file_paths : list
        List of audio file paths
    show_plot : bool
        Whether to show visualization
    """
    aggregator = TemporalAggregator(
        window_size=TEMPORAL_WINDOW_SIZE,
        vote_threshold=VOTE_THRESHOLD,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        cooldown=ALERT_COOLDOWN
    )
    
    predictions = []
    vote_counts = []
    alerts = []
    
    print(f"Testing on {len(file_paths)} files...\n")
    
    for i, file_path in enumerate(file_paths):
        if not Path(file_path).exists():
            print(f"⚠️ File not found: {file_path}")
            continue
        
        # Load and predict
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
        probability = predict_window(audio, model)
        
        # Add to aggregator
        result = aggregator.add_prediction(probability)
        
        predictions.append(probability)
        vote_counts.append(result['vote_count'])
        alerts.append(result['alert'])
        
        # Print result
        status = "🚨 ALERT" if result['alert'] else f"Votes: {result['vote_count']}/{TEMPORAL_WINDOW_SIZE}"
        print(f"[{i+1:3d}] {Path(file_path).name:40s} | Prob: {probability:.4f} | {status}")
    
    # Summary
    print(f"\nSummary:")
    print(f"  Files processed: {len(predictions)}")
    print(f"  Alerts triggered: {sum(alerts)}")
    print(f"  Avg probability: {np.mean(predictions):.4f}")
    print(f"  Max probability: {np.max(predictions):.4f}")
    print(f"  Min probability: {np.min(predictions):.4f}")
    
    # Plot
    if show_plot and predictions:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
        
        # Predictions over time
        ax1.plot(predictions, marker='o', label='Probability', linewidth=2)
        ax1.axhline(y=CONFIDENCE_THRESHOLD, color='r', linestyle='--', label=f'Threshold ({CONFIDENCE_THRESHOLD})')
        ax1.fill_between(range(len(predictions)), 0, 1, 
                         where=[a for a in alerts], alpha=0.3, color='red', label='Alert Triggered')
        ax1.set_xlabel('Window Index')
        ax1.set_ylabel('Probability')
        ax1.set_title('Predictions Over Time')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim([0, 1])
        
        # Vote counts
        ax2.bar(range(len(vote_counts)), vote_counts, alpha=0.7, label='Vote Count')
        ax2.axhline(y=VOTE_THRESHOLD, color='r', linestyle='--', label=f'Vote Threshold ({VOTE_THRESHOLD})')
        ax2.set_xlabel('Window Index')
        ax2.set_ylabel('Vote Count')
        ax2.set_title(f'Temporal Voting (Window Size: {TEMPORAL_WINDOW_SIZE})')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        ax2.set_ylim([0, TEMPORAL_WINDOW_SIZE + 1])
        
        plt.tight_layout()
        plt.show()

print("✓ Batch testing function defined")

✓ Batch testing function defined


In [ ]:
# Example: Test on first 20 files from training set
from pathlib import Path

train_files = sorted(Path('dataset/train').glob('*.wav'))[:20]

if train_files:
    print(f"Found {len(train_files)} files to test\n")
    test_on_files([str(f) for f in train_files], show_plot=True)
else:
    print("No files found in dataset/train/")
    print("Skipping batch test...")

## 10. Utilities & Configuration Helpers

In [ ]:
def list_available_models():
    """List all available models in the models/ folder."""
    models_dir = Path('models')
    if not models_dir.exists():
        print("❌ models/ folder not found")
        return
    
    model_files = list(models_dir.glob('*.pth'))
    
    if not model_files:
        print("❌ No .pth model files found in models/")
        return
    
    print("Available models:")
    print("="*70)
    for i, model_file in enumerate(model_files, 1):
        size_mb = model_file.stat().st_size / (1024 * 1024)
        print(f"{i}. {model_file.name:40s} ({size_mb:.2f} MB)")
    print("="*70)
    print(f"\nTo use a different model, change MODEL_PATH in the configuration cell.")

list_available_models()

In [ ]:
def print_current_config():
    """Print current configuration."""
    print("Current Configuration:")
    print("="*70)
    print(f"Model:")
    print(f"  Path: {MODEL_PATH}")
    print(f"  Device: {DEVICE}")
    print()
    print(f"Audio Processing:")
    print(f"  Sample Rate: {SAMPLE_RATE} Hz")
    print(f"  Window Duration: {WINDOW_DURATION}s")
    print(f"  Mel Bands: {N_MELS}")
    print(f"  FFT Size: {N_FFT}")
    print(f"  Hop Length: {HOP_LENGTH}")
    print()
    print(f"Temporal Aggregation:")
    print(f"  Confidence Threshold: {CONFIDENCE_THRESHOLD}")
    print(f"  Window Size: {TEMPORAL_WINDOW_SIZE} predictions")
    print(f"  Vote Threshold: {VOTE_THRESHOLD}/{TEMPORAL_WINDOW_SIZE} must agree")
    print(f"  Alert Cooldown: {ALERT_COOLDOWN}s")
    print("="*70)

print_current_config()

## 11. Quick Start Guide

### 🚀 To Run Live Inference:

```python
# Monitor for 60 seconds
live_inference(duration=60)

# Monitor indefinitely (Ctrl+C to stop)
live_inference(duration=None)
```

### 🔧 To Change Model:

1. Go to **Configuration** cell (Section 1)
2. Change `MODEL_PATH` to point to your new model
3. Re-run cells from Section 1 onwards

### ⚙️ To Adjust Sensitivity:

**More Sensitive** (catch more alarms, more false alarms):
```python
CONFIDENCE_THRESHOLD = 0.3  # Lower threshold
VOTE_THRESHOLD = 2          # Fewer votes needed
```

**Less Sensitive** (fewer false alarms, might miss some alarms):
```python
CONFIDENCE_THRESHOLD = 0.7  # Higher threshold
VOTE_THRESHOLD = 4          # More votes needed
```

### 📊 To Test on Files:

```python
# Test on specific files
files = ['dataset/train/file1.wav', 'dataset/train/file2.wav']
test_on_files(files)

# Test on all alarm files
alarm_files = sorted(Path('dataset/train').glob('*alarm*.wav'))
test_on_files([str(f) for f in alarm_files])
```

### 🎯 Recommended Settings:

**For Medical Use** (prioritize catching all alarms):
```python
CONFIDENCE_THRESHOLD = 0.3
VOTE_THRESHOLD = 2
TEMPORAL_WINDOW_SIZE = 5
```

**For General Use** (balanced):
```python
CONFIDENCE_THRESHOLD = 0.5
VOTE_THRESHOLD = 3
TEMPORAL_WINDOW_SIZE = 5
```

**For Low False Alarms** (very conservative):
```python
CONFIDENCE_THRESHOLD = 0.7
VOTE_THRESHOLD = 4
TEMPORAL_WINDOW_SIZE = 5
```